# Stage 2

In [ ]:
!pip install -q transformers sentence-transformers accelerate bitsandbytes

In [ ]:
import torch
import json
import numpy as np
import os
import gc
import argparse
from tqdm import tqdm
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
import random

# ================= 配置區域 =================
# 請將此處替換為您的 RAGTruth 資料集路徑
DATASET_DIR = "/kaggle/input/dataset-by-myself" 
SOURCE_FILE = "HalluRAG_with_chunks.json"

MODEL_KEY = "Llama-3.2-1B"
# MODEL_KEY = "Llama-3.2-3B"

if MODEL_KEY == "Llama-3.2-1B":
    COPY_FILE = "top_copy_head_pairs_1B.json"
elif MODEL_KEY == "Llama-3.2-3B":
    COPY_FILE = "top_copy_head_pairs_3B.json"

# 模型設定
MODEL_NAME = f"meta-llama/{MODEL_KEY}-Instruct"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# 使用 HuggingFace Token
HF_TOKEN = "hf_XXXXXXXXXXXX" 
login(HF_TOKEN)

# 輸出的檔案名稱
OUTPUT_FILE = f"{MODEL_KEY}_hallurag_results.json"

# 硬體設定
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 隨機變數
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# BATCH_SIZE
BATCH_SIZE = 25 # Hallurag

In [ ]:
# ================= 輔助計算函數 =================

def calculate_sentence_similarity(text1, text2, model):
    """計算語義相似度 (ECS)"""
    if not text1 or not text2: return 0.0
    emb1 = model.encode([text1], normalize_embeddings=True)
    emb2 = model.encode([text2], normalize_embeddings=True)
    return float(np.dot(emb1, emb2.T)[0,0])

def calculate_jsd(logits_p, logits_q):
    """
    計算 Jensen-Shannon Divergence (PKS) 
    """
    logits_p = logits_p.to(torch.float32)
    logits_q = logits_q.to(torch.float32)
    
    p = F.softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)
    
    m = 0.5 * (p + q)
    
    log_m = torch.log(m + 1e-20) 
    
    kl_p = F.kl_div(log_m, p, reduction='none').sum(dim=-1)
    kl_q = F.kl_div(log_m, q, reduction='none').sum(dim=-1)

    jsd = 0.5 * (kl_p + kl_q)

    if torch.isnan(jsd).any():
        jsd = torch.nan_to_num(jsd, nan=0.0)

    return jsd.mean().item()

def get_token_spans_from_char_spans(char_spans, offset_mapping):
    token_spans = []
    if not char_spans:
        return token_spans

    for sp in char_spans:
        if isinstance(sp, dict):
            c_start, c_end = sp['start'], sp['end']
        else:
            c_start, c_end = sp[0], sp[1]
            
        t_start = None
        t_end = None
        
        # 尋找對應的 token 範圍
        for idx, (tok_start_char, tok_end_char) in enumerate(offset_mapping):
            # 跳過特殊 token
            if tok_start_char == tok_end_char: continue
            
            # 判斷重疊：只要 token 與 char span 有交集就算
            if max(c_start, tok_start_char) < min(c_end, tok_end_char):
                if t_start is None: t_start = idx
                t_end = idx + 1 # end is exclusive
        
        if t_start is not None and t_end is not None:
            token_spans.append([t_start, t_end])
            
    return token_spans

def is_hallucination_span(r_span, hallucination_token_indices):
    for idx in range(r_span[0], r_span[1]):
        if idx in hallucination_token_indices:
            return 1
    return 0

def debug_alignment(
    tokenizer,
    full_text,
    input_ids,
    char_span,
    token_span,
    tag=""
):
    print("\n===== ALIGNMENT CHECK", tag, "=====")

    c_start, c_end = char_span
    t_start, t_end = token_span

    print("\n[1] full_text char slice:")
    print(repr(full_text[c_start:c_end]))

    print("\n[2] token decoded slice:")
    print(repr(tokenizer.decode(input_ids[0, t_start:t_end])))

    print(f"\n[3] char span = {char_span}")
    print(f"\n[4] token span = {token_span}")

# ================= 主流程 =================

def main():
    
    print(f"使用裝置: {DEVICE}")
    
    # 1. 載入模型 (Eager Mode)
    print(f"正在載入 LLM: {MODEL_NAME} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        # torch_dtype=torch.float32,
        token=HF_TOKEN,
        attn_implementation="eager" 
    )
    model.eval()

    # 2. 載入 Embedding 模型
    print(f"正在載入 Embedding Model: {EMBEDDING_MODEL_NAME} ...")
    bge_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE, token=HF_TOKEN)

    # 3. Copying Heads 偵測( Llama 3.2 1B )
    if MODEL_KEY == "Llama-3.2-1B":
        with open(os.path.join(DATASET_DIR, COPY_FILE),'r') as f:
            copy_heads = json.load(f)[:16]     
        KNOWLEDGE_LAYERS = list(range(16)) 
    elif MODEL_KEY == "Llama-3.2-3B":    
        with open(os.path.join(DATASET_DIR, COPY_FILE),'r') as f:
            copy_heads = json.load(f)[:28]
        KNOWLEDGE_LAYERS = list(range(28)) 

    # 4. 載入資料
    print("正在載入 Hallurag 資料集...")
    with open(os.path.join(DATASET_DIR, SOURCE_FILE), "r", encoding="utf-8") as f:
        responses_data = json.load(f)
    
    # 4.1. 依 split 拆分 train/test
    train_data = [x for x in responses_data if x["split"] == "train"]
    test_data  = [x for x in responses_data if x["split"] == "test"]
    
    print("原始 train:", len(train_data), "原始 test:", len(test_data))

    ratio = 1
    
    # 4.2. 各自做抽樣
    train_data = random.sample(train_data, int(ratio * len(train_data)))
    test_data  = random.sample(test_data, int(ratio * len(test_data)))
    
    print("抽樣後 train:", len(train_data), "抽樣後 test:", len(test_data))
    
    # 4.3. 合併回一個列表
    responses_data = train_data + test_data
    print("合併後總筆數:", len(responses_data))

    # 5. 主迴圈
    first_debug_print = True
    select_response = []
    for item in tqdm(responses_data):

        # 拿出我要的資料
        prompt = item["prompt_text"]
        response_text = item["response"]

        # offset 交換
        prompt_prefix = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        assistant_prefix = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        
        prompt_char_offset = len(prompt_prefix)
        response_char_offset = (
            len(prompt_prefix)
            + len(prompt)
            + len(assistant_prefix)
        )

        # 建構完整的 Chat 輸入
        full_text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response_text}<|eot_id|>"
        
        inputs = tokenizer(
            full_text, 
            return_tensors="pt", 
            return_offsets_mapping=True,
            add_special_tokens=False
        )
        input_ids = inputs.input_ids.to(DEVICE)
        offsets = inputs.offset_mapping[0].cpu().tolist()
        
        # 計算 Prefix Length (User Prompt 部分)
        prompt_only_text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        prompt_ids = tokenizer(prompt_only_text, return_tensors="pt", add_special_tokens=False).input_ids
        prefix_len = prompt_ids.shape[-1]
        
        if input_ids.shape[-1] <= prefix_len: continue

        # ===== 把 dataset span 轉成 full_text 座標 =====
        prompt_char_spans = [
            [s + prompt_char_offset, e + prompt_char_offset]
            for s, e in item.get("prompt_spans", [])
        ]
        
        response_char_spans = [
            [s + response_char_offset, e + response_char_offset]
            for s, e in item.get("response_spans", [])
        ]
        
        # ===== 再轉成 token span =====
        prompt_token_spans = get_token_spans_from_char_spans(
            prompt_char_spans, offsets
        )
        response_token_spans = get_token_spans_from_char_spans(
            response_char_spans, offsets
        )

        # ===== 取得 Hallucination Labels 的 Token Indices =====
        hallucination_token_indices = set()
        hs = item.get("hallucination_span")
        if hs is not None:
            h_ranges = get_token_spans_from_char_spans(
                [[
                    hs["start"] + response_char_offset,
                    hs["end"] + response_char_offset
                ]],
                offsets
            )
            for r in h_ranges:
                hallucination_token_indices.update(range(r[0], r[1]))

        if first_debug_print:
            # 檢查第一個 response chunk
            debug_alignment(
                tokenizer,
                full_text,
                input_ids,
                response_char_spans[0],
                response_token_spans[0],
                tag="RESPONSE"
            )
        
            # 檢查第一個 prompt span
            if prompt_char_spans:
                debug_alignment(
                    tokenizer,
                    full_text,
                    input_ids,
                    prompt_char_spans[0],
                    prompt_token_spans[0],
                    tag="PROMPT"
                )
        
            first_debug_print = False

        # --- 模型推論 ---
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                output_attentions=True,
                output_hidden_states=True,
                return_dict=True
            )
        
        span_score_dict = []
        
        # --- Chunk Level 分析 ---
        for r_span in response_token_spans:
            # 確保 span 在範圍內
            if r_span[0] >= input_ids.shape[-1]: continue
            
            chunk_text = tokenizer.decode(input_ids[0, r_span[0]:r_span[1]])
            
            # 1. ECS: Prompt Attention Score (針對每個 Copy Head)
            layer_head_span = {}
            for layer_idx, head_idx in copy_heads:
                attn_map = outputs.attentions[layer_idx][0, head_idx, r_span[0]:r_span[1], :prefix_len]
                
                if attn_map.numel() == 0: 
                    layer_head_span[str((layer_idx, head_idx))] = 0.0
                    continue
                
                # 尋找這個 Chunk 最關注哪一個 Prompt Span
                best_sim = 0.0
                
                # 計算每個 Prompt Span 的 Attention 總和
                p_span_scores = []
                for p_span in prompt_token_spans:
                    # 確保 p_span 在 prefix 範圍內
                    if p_span[1] > prefix_len: continue
                    score = attn_map[:, p_span[0]:p_span[1]].sum().item()
                    p_span_scores.append((p_span, score))
                
                if p_span_scores:
                    # 取出 Attention 最高的 Prompt Span
                    best_p_span = max(p_span_scores, key=lambda x: x[1])[0]
                    p_text = tokenizer.decode(input_ids[0, best_p_span[0]:best_p_span[1]])
                    
                    # 計算 BGE 相似度
                    best_sim = calculate_sentence_similarity(p_text, chunk_text, bge_model)
                
                layer_head_span[str((layer_idx, head_idx))] = best_sim

            # 2. PKS: Parameter Knowledge Scores (針對每個 Knowledge Layer)
            parameter_knowledge_dict = {}
            final_norm = model.model.norm
            lm_head = model.lm_head
            
            for layer_i in KNOWLEDGE_LAYERS:
                if layer_i + 1 >= len(outputs.hidden_states): break
                
                h_curr = outputs.hidden_states[layer_i][0, r_span[0]:r_span[1], :] # Input to Layer i
                h_next = outputs.hidden_states[layer_i+1][0, r_span[0]:r_span[1], :] #  Output of Layer i
                
                logits_curr = lm_head(final_norm(h_curr))
                logits_next = lm_head(final_norm(h_next))
                
                jsd_score = calculate_jsd(logits_curr, logits_next)
                parameter_knowledge_dict[f"layer_{layer_i}"] = jsd_score
            
            # 3. hallucination_Label 補回
            h_label = is_hallucination_span(r_span, hallucination_token_indices)
            
            # 建立輸出物品
            span_score_dict.append({
                "prompt_attention_score": layer_head_span,
                "r_span": r_span,
                "hallucination_label": h_label,
                "parameter_knowledge_scores": parameter_knowledge_dict
            })
            
        # 將計算結果加入原始資料
        item["scores"] = span_score_dict
        select_response.append(item)

        # 記憶體清理 (每 10 筆)，看硬體資源決定大小
        if len(select_response) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        # ============================================
        # 每滿 BATCH_SIZE 筆就寫入一次，看硬體資源決定大小
        # ============================================
        if len(select_response) >= BATCH_SIZE:
            print(f"\n>>> 已達 {BATCH_SIZE} 筆，正在 append 寫入 {OUTPUT_FILE} ...")
            with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                for obj in select_response:
                    f.write(json.dumps(obj, ensure_ascii=False) + "\n")

            select_response = []  # 清空 array

            gc.collect()
            torch.cuda.empty_cache()

    # ==========================
    # 處理剩下未滿 BATCH_SIZE 的
    # ==========================
    if len(select_response) > 0:
        print(f"\n>>> 寫入最後 {len(select_response)} 筆到 {OUTPUT_FILE} ...")
        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            for obj in select_response:
                f.write(json.dumps(obj, ensure_ascii=False) + "\n")

        select_response = []

        gc.collect()
        torch.cuda.empty_cache()

    print("全部完成！")

In [ ]:
if __name__ == "__main__":
    main()